In [ ]:
# Ячейка 1: Очистка порта и установка
!pip install -q fastapi uvicorn pyngrok nest-asyncio python-multipart httpx
!pip install -q diffusers transformers accelerate torch
!pip install -q xformers

# Убиваем процессы на порту 8000
!fuser -k 8000/tcp 2>/dev/null || true
import time
time.sleep(2)

In [ ]:
# Ячейка 2: Настройка ngrok
!pip install -q pyngrok
from pyngrok import ngrok
import os

# Вставь свой токен из ngrok.com
ngrok_token = "326I0SsywZZoSNLyXDtqVMiCAKk_7bFHa2odN7mRKQRzFxnAL"
!ngrok authtoken $ngrok_token

In [ ]:
# Ячейка 3: Загрузка модели
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline
import torch
import base64
from io import BytesIO
from PIL import Image
import nest_asyncio
import asyncio
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
import httpx
import threading
import time

MODEL_ID = "runwayml/stable-diffusion-v1-5"

print("📦 Загрузка модели...")
pipe_img2img = StableDiffusionImg2ImgPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
).to("cuda")

pipe_txt2img = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
).to("cuda")

print("✅ Модель загружена!")

In [ ]:
# Ячейка 4: FastAPI сервер
app = FastAPI()

@app.get("/")
async def root():
    return {"status": "online", "message": "Stable Diffusion API работает!"}

@app.post("/generate")
async def generate_interior(
    file: UploadFile = File(...),
    style: str = Form("modern")
):
    try:
        image_data = await file.read()
        image = Image.open(BytesIO(image_data))

        styles = {
            "modern": "Modern interior, clean lines, minimalist, neutral colors",
            "classic": "Classic interior, elegant, rich colors, ornate details",
            "minimalist": "Minimalist interior, simple, uncluttered, zen",
            "loft": "Industrial loft, exposed brick, urban style, open plan",
            "scandinavian": "Scandinavian, hygge, light wood, cozy, natural",
            "bohemian": "Bohemian, eclectic, colorful, vintage, artistic",
            "artdeco": "Art Deco, luxurious, geometric, bold colors",
            "japandi": "Japandi, Japanese-Scandinavian, minimal, natural"
        }

        prompt = styles.get(style, styles["modern"])
        full_prompt = f"{prompt}, interior design, photorealistic, 4k, professional photography"

        result = pipe_img2img(
            prompt=full_prompt,
            image=image,
            strength=0.75,
            num_inference_steps=30,
            guidance_scale=7.5,
            negative_prompt="low quality, blurry, ugly, distorted",
            num_images=4
        )

        images = []
        for i, img in enumerate(result.images):
            buffered = BytesIO()
            img.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode('utf-8')
            images.append(f"data:image/png;base64,{img_base64}")

        return {
            "success": True,
            "images": images,
            "style": style,
            "count": len(images)
        }

    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"success": False, "error": str(e)}
        )

@app.post("/inspire")
async def generate_inspiration(style: str = Form("modern")):
    try:
        styles = {
            "modern": "Modern interior, clean lines, minimalist, neutral colors",
            "classic": "Classic interior, elegant, rich colors, ornate details",
            "minimalist": "Minimalist interior, simple, uncluttered, zen",
            "loft": "Industrial loft, exposed brick, urban style, open plan",
            "scandinavian": "Scandinavian, hygge, light wood, cozy, natural",
            "bohemian": "Bohemian, eclectic, colorful, vintage, artistic",
            "artdeco": "Art Deco, luxurious, geometric, bold colors",
            "japandi": "Japandi, Japanese-Scandinavian, minimal, natural"
        }

        prompt = styles.get(style, styles["modern"])
        full_prompt = f"{prompt}, interior design, photorealistic, 4k, professional photography, wide angle view"

        result = pipe_txt2img(
            prompt=full_prompt,
            negative_prompt="low quality, blurry, ugly, distorted",
            num_inference_steps=30,
            guidance_scale=7.5,
            num_images=4
        )

        images = []
        for i, img in enumerate(result.images):
            buffered = BytesIO()
            img.save(buffered, format="PNG")
            img_base64 = base64.b64encode(buffered.getvalue()).decode('utf-8')
            images.append(f"data:image/png;base64,{img_base64}")

        return {
            "success": True,
            "images": images,
            "style": style,
            "count": len(images)
        }

    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"success": False, "error": str(e)}
        )

@app.get("/health")
async def health():
    return {"status": "healthy", "model": MODEL_ID}

In [ ]:
# Ячейка 5: Запуск сервера
import nest_asyncio
nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

time.sleep(3)
print("✅ Сервер запущен на порту 8000")

In [ ]:
# Ячейка 6: Создание публичного туннеля
import re

ngrok.kill()
tunnel = ngrok.connect(8000)
public_url_str = str(tunnel).replace('NgrokTunnel: "', '').replace('" -> "http://localhost:8000"', '')
print(f"🌐 Публичный адрес: {public_url_str}")

In [ ]:
# Ячейка 7: Проверка
import requests

try:
    response = requests.get(f"{public_url_str}/")
    print(f"✅ Тест: {response.json()}")
    print("\n🎉 Всё работает! Используй этот URL в бекенде:")
    print(f"   {public_url_str}")
except Exception as e:
    print(f"⚠️ Ошибка теста: {e}")
    print(f"Но сервер работает на: {public_url_str}")

%store public_url_str
print("\n📋 URL сохранен. Используй его в .env файле бекенда:")
print(f"SD_API_URL={public_url_str}")